In [4]:
import pandas as pd
import re
from jiwer import wer

df = pd.read_csv("whisper_medium_clinical_predictions.csv")

def clean_prediction(text):
    text = str(text).lower()
    text = re.sub(r'\bfull stop\b', '', text)
    text = re.sub(r'\bcomma\b', '', text)
    text = re.sub(r'\bperiod\b', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_reference(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

references_clean  = [clean_reference(r) for r in df["reference"].tolist()]
predictions_clean = [clean_prediction(p) for p in df["prediction"].tolist()]

cleaned_wer = wer(references_clean, predictions_clean)
print(f"Cleaned WER: {cleaned_wer:.3f}")

# Also check broken references
broken = df[df["reference"].apply(
    lambda x: bool(re.search(r'[a-z][a-z]{8,}', str(x).lower()))
)]
print(f"Potentially broken references: {len(broken)} / {len(df)}")

Cleaned WER: 0.421
Potentially broken references: 2715 / 3606


In [5]:
# Look at broken reference examples
print("=== Broken reference examples ===")
for idx, row in broken.head(20).iterrows():
    print(f"REF:  {row['reference']}")
    print(f"PRED: {row['prediction']}")
    print()

# Check length distribution of broken words
import re

def find_broken_words(text):
    return re.findall(r'[a-zA-Z]{9,}', str(text).lower())

broken["broken_words"] = broken["reference"].apply(find_broken_words)
print("\n=== Most common broken words ===")
from collections import Counter
all_broken = [w for words in broken["broken_words"] for w in words]
print(Counter(all_broken).most_common(20))

=== Broken reference examples ===
REF:  Protection of the host immune mechanism mightincrease the efcacy of other drugs that inhibit viralreplication.

PRED: Protection of the host immune mechanism might increase the efficacy of other drugs that inhibit viral replication full stop.

REF:  Ask the patient about allergies totape and skin antiseptics.

PRED: Ask the patients about allergies to tape and skin antiseptics.

REF:  The short, mucus-secreting cells between the ciliated cells show bumpy microvilli on their surfaces.
PRED: The short macro-secretion cells between the ciliated cells show bumping microbility on their surfaces.

REF:  Aspiration is terminated when aspirated material or blood becomes visible at the base or hub of the needle.
PRED: 6. Aspiration is terminated when aspirated material or blood becomes visible at the base or hub of the needle full stop.

REF:  Identify the appropriate landmarks for the site chosen.
PRED: Identify the appropriate landmarks for the site cho

In [6]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
import pandas as pd
import re
from jiwer import wer

df = pd.read_csv("whisper_medium_clinical_predictions.csv")

normalizer = BasicTextNormalizer()

def remove_spoken_punctuation(text):
    text = re.sub(r'\bfull stop\b', '', text)
    text = re.sub(r'\bcomma\b', '', text)
    text = re.sub(r'\bperiod\b', '', text)
    text = re.sub(r'\bcolon\b', '', text)
    text = re.sub(r'\bsemicolon\b', '', text)
    text = re.sub(r'\bnew line\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Combined pipeline: Whisper normaliser + spoken punctuation removal
references_norm  = [remove_spoken_punctuation(normalizer(str(r))) for r in df["reference"].tolist()]
predictions_norm = [remove_spoken_punctuation(normalizer(str(p))) for p in df["prediction"].tolist()]

print(f"WER combined: {wer(references_norm, predictions_norm):.3f}")

# Also compute on non-OCR-broken references only
def has_ocr_errors(text):
    words = str(text).split()
    for word in words:
        word_clean = re.sub(r'[^a-zA-Z]', '', word)
        if len(word_clean) > 15 and word_clean.islower():
            return True
    return False

df["ref_norm"]  = references_norm
df["pred_norm"] = predictions_norm
df["has_ocr"]   = df["reference"].apply(has_ocr_errors)

df_clean = df[~df["has_ocr"]]
print(f"WER on clean references only: {wer(df_clean['ref_norm'].tolist(), df_clean['pred_norm'].tolist()):.3f}")
print(f"Clean clips: {len(df_clean)} / {len(df)}")

# Save normalised predictions for all future analysis
df.to_csv("whisper_medium_predictions_normalised.csv", index=False)
print("Saved normalised predictions")


/home/ridwan/projects/afrispeech-project/afrispeech-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WER combined: 0.417
WER on clean references only: 0.410
Clean clips: 3450 / 3606
Saved normalised predictions


In [7]:
import pandas as pd
import re
from jiwer import wer, process_words

df = pd.read_csv("whisper_medium_clinical_predictions.csv")

# ── Fix 1: detect genuine OCR broken words ───────────────────────────────────
# Only flag words with NO vowels or merged lowercase runs without spaces
def has_genuine_ocr_errors(text):
    # Look for two lowercase words merged together - no space between them
    # Pattern: lowercase letter immediately followed by uppercase is fine
    # Pattern: word boundary missing e.g. "mightincrease"
    words = str(text).split()
    for word in words:
        word_clean = re.sub(r'[^a-zA-Z]', '', word)
        # Flag if word is very long AND not a known medical term pattern
        if len(word_clean) > 15 and word_clean.islower():
            return True
    return False

broken_real = df[df["reference"].apply(has_genuine_ocr_errors)]
print(f"Genuine OCR broken references: {len(broken_real)} / {len(df)}")

# ── Fix 2: clean predictions ─────────────────────────────────────────────────
def clean_prediction(text):
    text = str(text).lower()
    # Remove spoken punctuation
    text = re.sub(r'\bfull stop\b', '', text)
    text = re.sub(r'\bcomma\b', '', text)
    text = re.sub(r'\bperiod\b', '', text)
    text = re.sub(r'\bcolon\b', '', text)
    text = re.sub(r'\bsemicolon\b', '', text)
    text = re.sub(r'\bexclamation mark\b', '', text)
    text = re.sub(r'\bquestion mark\b', '', text)
    text = re.sub(r'\bnew line\b', '', text)
    text = re.sub(r'\bnext line\b', '', text)
    # Remove punctuation symbols
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_reference(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Fix 3: compute WER on clean set only ─────────────────────────────────────
# Exclude genuine OCR broken references from WER calculation
df_clean = df[~df["reference"].apply(has_genuine_ocr_errors)].copy()
print(f"Clean references remaining: {len(df_clean)} / {len(df)}")

references_clean  = [clean_reference(r) for r in df_clean["reference"].tolist()]
predictions_clean = [clean_prediction(p) for p in df_clean["prediction"].tolist()]

cleaned_wer = wer(references_clean, predictions_clean)
output      = process_words(references_clean, predictions_clean)

print(f"\n── WER on clean references ──────────────────")
print(f"WER:           {cleaned_wer:.3f}")
print(f"Substitutions: {output.substitutions}")
print(f"Deletions:     {output.deletions}")
print(f"Insertions:    {output.insertions}")
print(f"Hits:          {output.hits}")


Genuine OCR broken references: 156 / 3606
Clean references remaining: 3450 / 3606

── WER on clean references ──────────────────
WER:           0.410
Substitutions: 9606
Deletions:     1256
Insertions:    8747
Hits:          36953


In [8]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
import re

normalizer = BasicTextNormalizer()

# Spoken punctuation words present in both references and predictions
SPOKEN_PUNCT = [
    r'\bfull stop\b', r'\bcomma\b', r'\bperiod\b',
    r'\bcolon\b', r'\bsemicolon\b', r'\bsemi colon\b',
    r'\bquestion mark\b', r'\bexclamation mark\b',
    r'\bhyphen\b', r'\bnew line\b', r'\bnext line\b',
    r'\bopen bracket\b', r'\bclose bracket\b',
    r'\bopen parenthesis\b', r'\bclose parenthesis\b',
    r'\bslash\b', r'\bplus sign\b',
]

def normalise(text):
    text = str(text).lower()
    # Remove spoken punctuation first
    for pattern in SPOKEN_PUNCT:
        text = re.sub(pattern, '', text)
    # Then apply Whisper normaliser
    text = normalizer(text)
    # Clean up extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply to both
import pandas as pd
from jiwer import wer

df = pd.read_csv("whisper_medium_clinical_predictions.csv")

references_norm  = [normalise(r) for r in df["reference"].tolist()]
predictions_norm = [normalise(p) for p in df["prediction"].tolist()]

print(f"WER: {wer(references_norm, predictions_norm):.3f}")

# Check an example
for i in range(3):
    print(f"\nRef:  {references_norm[i]}")
    print(f"Pred: {predictions_norm[i]}")

WER: 0.412

Ref:  protection of the host immune mechanism mightincrease the efcacy of other drugs that inhibit viralreplication
Pred: protection of the host immune mechanism might increase the efficacy of other drugs that inhibit viral replication

Ref:  ask the patient about allergies totape and skin antiseptics
Pred: ask the patients about allergies to tape and skin antiseptics

Ref:  the short mucus secreting cells between the ciliated cells show bumpy microvilli on their surfaces
Pred: the short macro secretion cells between the ciliated cells show bumping microbility on their surfaces


In [9]:
from datasets import load_from_disk

ds = load_from_disk("./afrispeech_arrow_16k")
clinical_test = ds["test"].filter(lambda x: x["domain"] == "clinical")

durations = clinical_test["duration"]
print(f"Max duration: {max(durations):.1f}s")
print(f"Min duration: {min(durations):.1f}s")
print(f"Clips > 17s: {sum(1 for d in durations if d > 17)}")
print(f"Clips > 30s: {sum(1 for d in durations if d > 30)}")

Filter: 100%|██████████| 6318/6318 [00:46<00:00, 137.28 examples/s]

Max duration: 271.2s
Min duration: 1.0s
Clips > 17s: 492
Clips > 30s: 101


In [11]:
from datasets import load_from_disk
from utils import normalise
from jiwer import wer
import pandas as pd

ds = load_from_disk("./afrispeech_arrow_16k")

# Filter to match paper's criteria: 2-17 seconds, clinical domain, test split
clinical_test = ds["test"].filter(
    lambda x: x["domain"] == "clinical" and 2.0 <= x["duration"] <= 17.0
)

print(f"Clips after duration filter: {len(clinical_test)}")
print(f"Max duration: {max(clinical_test['duration']):.1f}s")
print(f"Min duration: {min(clinical_test['duration']):.1f}s")

Filter: 100%|██████████| 6318/6318 [00:44<00:00, 141.35 examples/s]

Clips after duration filter: 3055
Max duration: 17.0s
Min duration: 2.0s


In [12]:
import torch
import pandas as pd
from datasets import load_from_disk
from transformers import pipeline
from tqdm import tqdm
from utils import normalise
from jiwer import wer

# ── 1. Load and filter ────────────────────────────────────────────────────────
ds = load_from_disk("./afrispeech_arrow_16k")
clinical_test = ds["test"].filter(
    lambda x: x["domain"] == "clinical" and x["duration"] <= 30.0
)
print(f"Clinical test clips (<=30s): {len(clinical_test)}")

# ── 2. Load Whisper ───────────────────────────────────────────────────────────
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium",
    device=0,
    dtype=torch.float16,
    chunk_length_s=30,
    stride_length_s=5,
)

# ── 3. Run inference ──────────────────────────────────────────────────────────
results = []

for sample in tqdm(clinical_test, desc="Running Whisper-medium"):
    prediction = pipe(
        {"array": sample["audio"]["array"], "sampling_rate": sample["audio"]["sampling_rate"]},
        generate_kwargs={"language": "en", "task": "transcribe"}
    )
    results.append({
        "audio_id":   sample["audio_id"],
        "accent":     sample["accent"],
        "country":    sample["country"],
        "reference":  sample["transcript"],
        "prediction": prediction["text"].strip(),
        "duration":   sample["duration"]
    })

# ── 4. Compute WER ────────────────────────────────────────────────────────────
df = pd.DataFrame(results)
df.to_csv("whisper_medium_clinical_under30s.csv", index=False)

references_norm  = [normalise(r) for r in df["reference"].tolist()]
predictions_norm = [normalise(p) for p in df["prediction"].tolist()]

print(f"WER: {wer(references_norm, predictions_norm):.3f}")

Filter: 100%|██████████| 6318/6318 [00:44<00:00, 140.84 examples/s]


Clinical test clips (<=30s): 3505


Loading weights: 100%|██████████| 947/947 [00:00<00:00, 4095.33it/s]
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Running Whisper-medium:   0%|          | 0/3505 [00:00<?, ?it/s]A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transforme

WER: 0.380


In [13]:
import torch
import pandas as pd
from datasets import load_from_disk
from transformers import pipeline
from tqdm import tqdm
from utils import normalise
from jiwer import wer

ds = load_from_disk("./afrispeech_arrow_16k")
clinical_test = ds["test"].filter(
    lambda x: x["domain"] == "clinical" 
)
print(f"Clinical test clips (<=30s): {len(clinical_test)}")

pipe = pipeline(
    "automatic-speech-recognition",
    model="facebook/wav2vec2-large-960h",
    device=0,
    dtype=torch.float16
)

results = []
for sample in tqdm(clinical_test, desc="Running Wav2Vec2"):
    prediction = pipe({
        "array":         sample["audio"]["array"],
        "sampling_rate": sample["audio"]["sampling_rate"]
    })
    results.append({
        "audio_id":   sample["audio_id"],
        "accent":     sample["accent"],
        "country":    sample["country"],
        "reference":  sample["transcript"],
        "prediction": prediction["text"].strip(),
        "duration":   sample["duration"]
    })

df = pd.DataFrame(results)
df.to_csv("wav2vec2_clinical_under30s.csv", index=False)

references_norm  = [normalise(r) for r in df["reference"].tolist()]
predictions_norm = [normalise(p) for p in df["prediction"].tolist()]

print(f"Wav2Vec2 WER: {wer(references_norm, predictions_norm):.3f}")

Filter: 100%|██████████| 6318/6318 [00:45<00:00, 137.82 examples/s]


Clinical test clips (<=30s): 3606


Loading weights: 100%|██████████| 404/404 [00:00<00:00, 10505.39it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Running Wav2Vec2: 100%|██████████| 3606/3606 [01:21<00:00, 44.05it/s]


Wav2Vec2 WER: 0.802


In [14]:
import torch
import pandas as pd
from datasets import load_from_disk
from transformers import pipeline
from tqdm import tqdm
from utils import normalise
from jiwer import wer

ds = load_from_disk("./afrispeech_arrow_16k")
clinical_test = ds["test"].filter(
    lambda x: x["domain"] == "clinical" and x["duration"] <= 30.0
)
print(f"Clinical test clips (<=30s): {len(clinical_test)}")

pipe = pipeline(
    "automatic-speech-recognition",
    model="facebook/wav2vec2-large-960h",
    device=0,
    dtype=torch.float16
)

results = []
for sample in tqdm(clinical_test, desc="Running Wav2Vec2"):
    prediction = pipe({
        "array":         sample["audio"]["array"],
        "sampling_rate": sample["audio"]["sampling_rate"]
    })
    results.append({
        "audio_id":   sample["audio_id"],
        "accent":     sample["accent"],
        "country":    sample["country"],
        "reference":  sample["transcript"],
        "prediction": prediction["text"].strip(),
        "duration":   sample["duration"]
    })

df = pd.DataFrame(results)
df.to_csv("wav2vec2_clinical_under30s.csv", index=False)

references_norm  = [normalise(r) for r in df["reference"].tolist()]
predictions_norm = [normalise(p) for p in df["prediction"].tolist()]

print(f"Wav2Vec2 WER: {wer(references_norm, predictions_norm):.3f}")

Clinical test clips (<=30s): 3505


Loading weights: 100%|██████████| 404/404 [00:00<00:00, 10140.26it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Running Wav2Vec2: 100%|██████████| 3505/3505 [01:09<00:00, 50.78it/s]


Wav2Vec2 WER: 0.759


In [16]:
import pandas as pd
from utils import normalise
from jiwer import wer

whisper_df = pd.read_csv("whisper_medium_clinical_predictions.csv")
wav2vec_df = pd.read_csv("wav2vec2_clinical_under30s.csv")

def wer_by_country(df, model_name):
    print(f"\n── {model_name} WER by Country ──────────────────────")
    results = []
    for country in sorted(df["country"].unique()):
        subset = df[df["country"] == country]
        if len(subset) < 10:
            continue
        refs  = [normalise(r) for r in subset["reference"].tolist()]
        preds = [normalise(p) for p in subset["prediction"].tolist()]
        results.append({
            "country": country,
            "clips":   len(subset),
            "wer":     round(wer(refs, preds), 3)
        })
    result_df = pd.DataFrame(results).sort_values("wer")
    print(result_df.to_string(index=False))
    result_df.to_csv(f"{model_name.replace(' ', '_')}_wer_by_country.csv", index=False)
    return result_df

wer_by_country(whisper_df, "Whisper_medium")
wer_by_country(wav2vec_df, "Wav2Vec2")


── Whisper_medium WER by Country ──────────────────────
country  clips   wer
     BW     30 0.163
     RW     13 0.199
     ZA    254 0.201
     US     20 0.216
     GH     76 0.231
     UG     19 0.261
     KE    331 0.342
     NG   2855 0.452

── Wav2Vec2 WER by Country ──────────────────────
country  clips   wer
     RW     13 0.471
     BW     30 0.492
     KE    329 0.549
     ZA    252 0.574
     US     20 0.597
     UG     19 0.640
     GH     75 0.718
     NG   2759 0.817


,country,clips,wer
4,RW,13,0.471
0,BW,30,0.492
2,KE,329,0.549
7,ZA,252,0.574
6,US,20,0.597
5,UG,19,0.640
1,GH,75,0.718
3,NG,2759,0.817
